In [22]:
with open("/Users/anand/Desktop/LLM From Scratch/Dataset/the-verdict.txt" , "r") as f:
    raw_text = f.read()
    
print("The total length of the text is : ", len(raw_text))
print(raw_text[:99])  # Print the first 500 characters of the text

The total length of the text is :  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [16]:
import re 

text = "Hello ,  world! This , is a test. Let's see: how well does it work?"
result = re.split(r'(\s)' , text)

print(result)

['Hello', ' ', ',', ' ', '', ' ', 'world!', ' ', 'This', ' ', ',', ' ', 'is', ' ', 'a', ' ', 'test.', ' ', "Let's", ' ', 'see:', ' ', 'how', ' ', 'well', ' ', 'does', ' ', 'it', ' ', 'work?']


In [17]:
new_res = re.split(r'([,.]|\s)' , text)
print(new_res)

['Hello', ' ', '', ',', '', ' ', '', ' ', 'world!', ' ', 'This', ' ', '', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '', ' ', "Let's", ' ', 'see:', ' ', 'how', ' ', 'well', ' ', 'does', ' ', 'it', ' ', 'work?']


In [18]:
results = [list for list in new_res if list.strip()]
print(results)

['Hello', ',', 'world!', 'This', ',', 'is', 'a', 'test', '.', "Let's", 'see:', 'how', 'well', 'does', 'it', 'work?']


In [20]:
text_new = "Hello , world! Is this-- a test?"
res = re.split(r'([,.:;"()\!]|--|\s)', text_new)
res = [item.strip() for item in res if item.strip()]
print(res)


['Hello', ',', 'world', '!', 'Is', 'this', '--', 'a', 'test?']


In [27]:
preprocessed_text = re.split(r'([,.:;"()\!]|--|\s)', raw_text)
preprocessed_text = [token.strip() for token in preprocessed_text if token.strip()]
print(preprocessed_text[:99])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter']


In [24]:
print(len(preprocessed_text))

4459


In [ ]:
import json
import tokenize
import token
import keyword
from io import BytesIO
from typing import List


# --------------------------------------------------
# 1. Load JSON file (array-based JSON)
# --------------------------------------------------
def load_code_samples(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    codes = []
    for item in data:
        if "code" in item:
            codes.append(item["code"])

    return codes


# --------------------------------------------------
# 2. Python semantic tokenizer (indentation-aware)
# --------------------------------------------------
def python_semantic_tokenizer(code: str) -> List[str]:
    tokens = []

    for tok in tokenize.tokenize(BytesIO(code.encode("utf-8")).readline):
        ttype = tok.type
        tval = tok.string

        if ttype == token.INDENT:
            tokens.append("<INDENT>")

        elif ttype == token.DEDENT:
            tokens.append("<DEDENT>")

        elif ttype == token.NEWLINE:
            tokens.append("<NEWLINE>")

        elif ttype == token.NAME:
            if keyword.iskeyword(tval):
                tokens.append(tval)
            else:
                tokens.append("<IDENT>")

        elif ttype == token.NUMBER:
            tokens.append("<NUMBER>")

        elif ttype == token.STRING:
            tokens.append("<STRING>")

        elif ttype == token.OP:
            tokens.append(tval)

        elif ttype == token.ENDMARKER:
            tokens.append("<EOF>")

        # Ignore comments, encoding tokens, NL, etc.

    return tokens


# --------------------------------------------------
# 3. Tokenize entire dataset
# --------------------------------------------------
def tokenize_dataset(json_path: str):
    codes = load_code_samples(json_path)

    for idx, code in enumerate(codes, start=1):
        print(f"\n================ SAMPLE {idx} ================\n")
        print("Original Code:\n")
        print(code)

        tokens = python_semantic_tokenizer(code)

        print("\nTokenized Output:\n")
        print(" ".join(tokens))


# --------------------------------------------------
# 4. Entry point
# --------------------------------------------------
if __name__ == "__main__":
    JSON_PATH = "/Users/anand/Desktop/LLM From Scratch/Dataset/python_sample.json"
    tokenize_dataset(JSON_PATH)



================ SAMPLE 1 ================

Original Code:

def add(a, b):
    return a + b

Tokenized Output:

def <IDENT> ( <IDENT> , <IDENT> ) : <NEWLINE> <INDENT> return <IDENT> + <IDENT> <NEWLINE> <DEDENT> <EOF>

================ SAMPLE 2 ================

Original Code:

def is_even(n):
    return n % 2 == 0

Tokenized Output:

def <IDENT> ( <IDENT> ) : <NEWLINE> <INDENT> return <IDENT> % <NUMBER> == <NUMBER> <NEWLINE> <DEDENT> <EOF>

================ SAMPLE 3 ================

Original Code:

def factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

Tokenized Output:

def <IDENT> ( <IDENT> ) : <NEWLINE> <INDENT> <IDENT> = <NUMBER> <NEWLINE> for <IDENT> in <IDENT> ( <NUMBER> , <IDENT> + <NUMBER> ) : <NEWLINE> <INDENT> <IDENT> *= <IDENT> <NEWLINE> <DEDENT> return <IDENT> <NEWLINE> <DEDENT> <EOF>

================ SAMPLE 4 ================

Original Code:

def reverse_string(s):
    return s[::-1]

Tokenized Output:

def <IDENT> ( <IDENT